## Evaluate Generative Retrieval Model (T5)

Evaluates the trained T5 seq2seq model with beam search retrieval.

**What this notebook does:**
1. Load trained model checkpoint
2. Beam-search decode top-K Semantic IDs per user
3. Map Semantic IDs to items, filter invalid IDs
4. Compute Recall@K, NDCG@K on val + test splits
5. Compare against paper Table 1 (Toys & Games)

In [ ]:
import sys

if "../" not in sys.path:
    sys.path.insert(0, "../")

from pathlib import Path

import torch
from transformers import T5ForConditionalGeneration

from tiger.dataset import TigerDataset
from tiger.evaluation import evaluate
from tiger.utils import get_device, set_seed

In [2]:
# Paths
DATA_DIR = Path("../data/2014/processed")
SPLITS_PATH = DATA_DIR / "splits.parquet"
SEMANTIC_IDS_PATH = Path("../checkpoints/rqvae/semantic_ids.pt")
CHECKPOINT_PATH = Path("../checkpoints/model_adamw_3e-4/checkpoint-100000")

# Evaluation config
BATCH_SIZE = 64  # beam search is memory-heavy; keep modest
BEAM_SIZE = 20  # matches paper's Fig. 6 (top-20 retrieval)
AT_K = [5, 10]  # paper evaluates K = 5, 10

SEED = 42

In [3]:
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

2026-09-15 21:41:25.461 | INFO     | tiger.utils:set_seed:27 - Random seed set to 42
2026-09-15 21:41:25.478 | INFO     | tiger.utils:get_device:16 - Using device: mps


Device: mps


Load sid_to_asin mapping

In [5]:
sid_data = torch.load(SEMANTIC_IDS_PATH, weights_only=False)
sid_to_asin = sid_data["sid_to_asin"]

print(f"Items with Semantic IDs: {len(sid_to_asin):,}")

Items with Semantic IDs: 11,924


Load trained T5 model from checkpoint

In [6]:
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT_PATH).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading weights:   0%|          | 0/89 [00:00<?, ?it/s]

Total parameters: 14,540,160


### Evaluate Validation Split

Val users have their full training history as input and the held-out
second-to-last item as target.

In [7]:
val_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="val",
    max_seq_len=20,
)

print(f"Val users: {len(val_dataset):,}")

val_results = evaluate(
    model=model,
    dataset=val_dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=BATCH_SIZE,
    beam_size=BEAM_SIZE,
    at_k=AT_K,
)

print("\nVal results:")
for k, v in val_results.items():
    print(f"  {k}: {v:.4f}")

Val users: 19,412


Evaluating: 100%|██████████| 304/304 [01:10<00:00,  4.34it/s]


Val results:
  recall@5: 0.0299
  ndcg@5: 0.0190
  recall@10: 0.0478
  ndcg@10: 0.0248


### Evaluate Test Split

Test users have training history + val item as input and the held-out
last item as target. This matches the paper's final reported numbers.

In [8]:
test_dataset = TigerDataset(
    splits_path=SPLITS_PATH,
    semantic_ids_path=SEMANTIC_IDS_PATH,
    split="test",
    max_seq_len=20,
)

print(f"Test users: {len(test_dataset):,}")

test_results = evaluate(
    model=model,
    dataset=test_dataset,
    sid_to_asin=sid_to_asin,
    device=device,
    batch_size=BATCH_SIZE,
    beam_size=BEAM_SIZE,
    at_k=AT_K,
)

print("\nTest results:")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}")

Test users: 19,412


Evaluating: 100%|██████████| 304/304 [01:10<00:00,  4.34it/s]


Test results:
  recall@5: 0.0206
  ndcg@5: 0.0127
  recall@10: 0.0335
  ndcg@10: 0.0168


### Compare with Paper (Toys & Games, Table 1)

In [9]:
import pandas as pd

paper = {
    "recall@5": 0.0521,
    "ndcg@5": 0.0371,
    "recall@10": 0.0712,
    "ndcg@10": 0.0432,
}

paper_seeds = {
    "recall@5": (0.0518, 0.00064),
    "ndcg@5": (0.0375, 0.00039),
    "recall@10": (0.0698, 0.0013),
    "ndcg@10": (0.0433, 0.00047),
}

sasrec_best_baseline = {
    "recall@5": 0.0463,
    "ndcg@5": 0.0306,
    "recall@10": 0.0675,
    "ndcg@10": 0.0374,
}

df = pd.DataFrame(
    {
        "Ours (val)": val_results,
        "Ours (test)": test_results,
        "Paper TIGER": paper,
        "Paper (3 seeds)": {
            k: f"{m:.4f} ± {s:.4f}" for k, (m, s) in paper_seeds.items()
        },
        "SASRec baseline": sasrec_best_baseline,
    }
)
df.index.name = "Metric"
df.round(4)

,Ours (val),Ours (test),Paper TIGER,Paper (3 seeds),SASRec baseline
Metric,,,,,
recall@5,0.0299,0.0206,0.0521,0.0518 ± 0.0006,0.0463
ndcg@5,0.0190,0.0127,0.0371,0.0375 ± 0.0004,0.0306
recall@10,0.0478,0.0335,0.0712,0.0698 ± 0.0013,0.0675
ndcg@10,0.0248,0.0168,0.0432,0.0433 ± 0.0005,0.0374
